# CineMatch – Professional UI su Google Colab

Questo notebook avvia la versione grafica evoluta del recommender MovieLens.

### Prima di iniziare
Per visualizzare **poster e trama**, crea una chiave gratuita su TMDb e usa:
- `TMDB_BEARER_TOKEN` oppure
- `TMDB_API_KEY`.

Il recommender continua a funzionare anche senza TMDb, ma le card useranno un placeholder.

In [1]:
!pip install -q streamlit numpy pandas scipy scikit-learn requests

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

print("Dipendenze installate.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 57.4 MB/s eta 0:00:00
Dipendenze installate.


In [2]:
!pip install -q streamlit pandas numpy scipy scikit-learn requests altair

## Inserisci la credenziale TMDb

Puoi salvare il token nei **Secrets di Colab** con il nome `TMDB_BEARER_TOKEN`.
Se non è presente, la cella te lo chiederà senza mostrarlo a schermo.

In [3]:
import os
from getpass import getpass

try:
    from google.colab import userdata
    token = userdata.get("TMDB_BEARER_TOKEN")
except Exception:
    token = ""

if not token:
    token = getpass("Incolla TMDB API Read Access Token (Bearer): ").strip()

if token:
    os.environ["TMDB_BEARER_TOKEN"] = token
    print("Token TMDb configurato.")
else:
    print("Nessun token: il recommender funzionerà, ma senza poster/trame TMDb.")

Token TMDb configurato.


In [4]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
python_files = [name for name in uploaded if name.lower().endswith('.py')]

if len(python_files) != 1:
    raise ValueError('Carica esattamente un file .py dell’app CineMatch.')

APP_PATH = Path('/content') / python_files[0]
print(f'App selezionata: {APP_PATH.name}')

Saving app.py to app.py
App selezionata: app.py


In [5]:
import os
import subprocess
import time

import requests

# Se riesegui la cella, chiude soltanto i processi avviati dal notebook.
for process_name in ('streamlit_process', 'tunnel_process'):
    process = globals().get(process_name)
    if process and process.poll() is None:
        process.terminate()
        process.wait(timeout=5)

streamlit_log_path = '/content/streamlit.log'
streamlit_log = open(streamlit_log_path, 'w')
streamlit_process = subprocess.Popen(
    [
        'streamlit', 'run', str(APP_PATH),
        '--server.address', '127.0.0.1',
        '--server.port', '8501',
        '--server.headless', 'true',
    ],
    stdout=streamlit_log,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)

health_url = 'http://127.0.0.1:8501/_stcore/health'
for _ in range(30):
    if streamlit_process.poll() is not None:
        break
    try:
        if requests.get(health_url, timeout=1).status_code == 200:
            print('Streamlit è pronto sulla porta 8501.')
            break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError('Streamlit non è diventato disponibile entro 30 secondi.')

if streamlit_process.poll() is not None:
    print(Path(streamlit_log_path).read_text())
    raise RuntimeError('Streamlit si è arrestato: leggi il log qui sopra.')

Streamlit è pronto sulla porta 8501.


In [ ]:
!./cloudflared tunnel --url http://127.0.0.1:8501

2026-10-17T09:14:04Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-17T09:14:04Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-10-17T09:14:07Z INF +--------------------------------------------------------------------------------------------+
2026-10-17T09:14:07Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-17T09:14:07Z INF |  https://unit-sauce-exploring-appearing.trycloudflare.